In [11]:
# First up, importing the essentials. 
# Pandas for the heavy lifting, and sqlite3 for that SQL file trick.
import pandas as pd
import sqlite3
import json

# Just setting this so I can actually see all the columns when I print the head.
pd.set_option('display.max_columns', None)

print("Setup done")

Setup done


In [14]:
# Step 1: Grabbing the transactional data from CSV.
df_orders = pd.read_csv('orders.csv')

# Step 2: Grabbing the user master data from JSON.
df_users = pd.read_json('users.json')

# Step 3: Now the sql file, as it's a .sql file, I'll run it in 
# a temporary memory database to turn it into a DataFrame.
conn = sqlite3.connect(':memory:')
with open('restaurants.sql', 'r') as f:
    sql_script = f.read()
conn.executescript(sql_script)

# Pulling the restaurants table into pandas.
df_restaurants = pd.read_sql_query("SELECT * FROM restaurants", conn)
conn.close()

print("All files loaded")
# Quick peek at the data
print(df_orders.head)

All files loaded
<bound method NDFrame.head of       order_id  user_id  restaurant_id  order_date  total_amount  \
0            1     2508            450  18-02-2023        842.97   
1            2     2693            309  18-01-2023        546.68   
2            3     2084            107  15-07-2023        163.93   
3            4      319            224  04-10-2023       1155.97   
4            5     1064            293  25-12-2023       1321.91   
...        ...      ...            ...         ...           ...   
9995      9996     2528            249  21-05-2023       1211.96   
9996      9997     2867            267  06-08-2023       1188.05   
9997      9998      522            420  11-11-2023        979.44   
9998      9999      319            492  08-09-2023       1105.93   
9999     10000      457            439  21-10-2023        879.58   

                     restaurant_name  
0                  New Foods Chinese  
1     Ruchi Curry House Multicuisine  
2              Spic

In [17]:
# Step 4: combine everything. 
# 'Left Join' 
# First, linking orders to the users who placed them.
merged_step1 = pd.merge(df_orders, df_users, on='user_id', how='left')

# Second, linking that result to the restaurant details.
final_df = pd.merge(merged_step1, df_restaurants, on='restaurant_id', how='left')

# Saving the final 'Source of Truth' as requested.
final_df.to_csv('final_food_delivery_dataset.csv', index=False)

print(f"Merge done. Total {len(final_df)} rows in the final dataset.")

Merge done. Total 10000 rows in the final dataset.


In [19]:
#1st MCQ
# Filter Gold
gold_df = final_df[final_df['membership'] == 'Gold']
# Get highest
top_city = gold_df.groupby('city')['total_amount'].sum().idxmax()
print(top_city)

Chennai


In [20]:
#2nd MCQ
# Group by cuisine and find highest average
top_cuisine = final_df.groupby('cuisine')['total_amount'].mean().idxmax()
print(top_cuisine)

Mexican


In [22]:
#3rd MCQ
# Filter Gold then find highest average city
# Sum total_amount by user
user_totals = final_df.groupby('user_id')['total_amount'].sum()
# Count users exceeding 1000
count_high_value = (user_totals > 1000).sum()
print(count_high_value)

2544


In [38]:
#4th MCQ
# Create bins
bins = [3.0, 3.5, 4.0, 4.5, 5.0]
labels = ['3.0 - 3.5', '3.6 - 4.0', '4.1 - 4.5', '4.6 - 5.0']
# Map ranges
final_df['rating_range'] = pd.cut(final_df['rating'], bins=bins, labels=labels, include_lowest=True)
# Get top
top_rating = final_df.groupby('rating_range')['total_amount'].sum().idxmax()
print(top_rating)

4.6 - 5.0


In [39]:
#5th MCQ
# Filter Gold members
gold_df = final_df[final_df['membership'] == 'Gold'] 
# Get city with highest mean
top_gold_city = gold_df.groupby('city')['total_amount'].mean().idxmax()
print(top_gold_city)

Chennai


In [40]:
#6th MCQ
# Count distinct restaurants and sum revenue by cuisine
cuisine_stats = final_df.groupby('cuisine').agg({'restaurant_id': 'nunique', 'total_amount': 'sum'}).sort_values('restaurant_id')
# Display the table
print(cuisine_stats)

         restaurant_id  total_amount
cuisine                             
Chinese            120    1930504.65
Indian             126    1971412.58
Italian            126    2024203.80
Mexican            128    2085503.09


In [41]:
#7th MCQ
# Calculate percentage
gold_count = len(final_df[final_df['membership'] == 'Gold'])
total_count = len(final_df)
pct_gold = round((gold_count / total_count) * 100)
print(f"{pct_gold}%")

50%


In [42]:
#8th MCQ
options = ['Grand Cafe Punjabi', 'Grand Restaurant South Indian', 'Ruchi Mess Multicuisine', 'Ruchi Foods Chinese']

# Filter stats for only these four
stats = final_df.groupby('restaurant_name_x').agg({'total_amount': 'mean', 'order_id': 'count'})
print(stats[stats.index.isin(options)])

                               total_amount  order_id
restaurant_name_x                                    
Grand Cafe Punjabi               765.409063        32
Grand Restaurant South Indian    842.567586        29
Ruchi Foods Chinese              686.603158        19
Ruchi Mess Multicuisine          851.226250        40


In [43]:
#9th MCQ
mcq_options = [('Gold', 'Indian'), ('Gold', 'Italian'), ('Regular', 'Indian'), ('Regular', 'Chinese')]
# Calculate and filter
revenue_stats = final_df.groupby(['membership', 'cuisine'])['total_amount'].sum()
highest_comb = revenue_stats[revenue_stats.index.isin(mcq_options)].idxmax()
print(highest_comb)

('Gold', 'Italian')


In [44]:
#10th MCQ
final_df['order_date'] = pd.to_datetime(final_df['order_date'], dayfirst=True)
# Find quarter with highest revenue
top_quarter = final_df.groupby(final_df['order_date'].dt.to_period('Q'))['total_amount'].sum().idxmax()
print(top_quarter)

2023Q3


In [45]:
#1st Numeric Entry
# Filter Gold and count total rows
gold_orders_count = len(final_df[final_df['membership'] == 'Gold'])
print(gold_orders_count)

4987


In [50]:
#2nd Numeric Entry
# Filter Hyderabad, sum revenue, and round
hyd_revenue = round(final_df[final_df['city'] == 'Hyderabad']['total_amount'].sum())
print(hyd_revenue)

1889367


In [51]:
#3rd Numeric Entry
# Count unique users
distinct_users = final_df['user_id'].nunique()
print(distinct_users)

2883


In [52]:
#4th Numeric Entry
# Filter Gold members
gold_orders = final_df[final_df['membership'] == 'Gold']
# Calculate average and round
avg_gold_order = gold_orders['total_amount'].mean()
print(round(avg_gold_order, 2))

797.15


In [53]:
#5th Numeric Entry
# Filter rating >= 4.5 and count
high_rating_orders = len(final_df[final_df['rating'] >= 4.5])
print(high_rating_orders)

3374


In [55]:
#6th Numeric Entry
# 1. Filter for Gold members
gold_df = final_df[final_df['membership'] == 'Gold']
# 2. Find the city with the highest total revenue
top_city = gold_df.groupby('city')['total_amount'].sum().idxmax() 
# 3. Count orders for that city among Gold members
order_count = len(gold_df[gold_df['city'] == top_city])
print(f"Order Count: {order_count}")

Order Count: 1337
